In [24]:
import json
import os
import urllib
import torch
from torch.utils.data import Dataset, DataLoader

In [25]:
from datasets import load_dataset

# Main community mirror (works with datasets>=3)
ds = load_dataset("tatsu-lab/alpaca")
print(ds)
print(ds["train"][0])

/home/ramsred/mmml/mmml-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


DatasetDict({
    train: Dataset({
        features: ['instruction', 'input', 'output', 'text'],
        num_rows: 52002
    })
})
{'instruction': 'Give three tips for staying healthy.', 'input': '', 'output': '1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.', 'text': 'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1.Eat a balanced diet and make sure to include plenty of fruits and vegetables. \n2. Exercise regularly to keep your body active and strong. \n3. Get enough sleep and maintain a consistent sleep schedule.'}


In [26]:
# First split: 50% train, 50% temp (to later become val+test)
train_test_split = ds["train"].train_test_split(test_size=0.5, seed=42)
train_ds = train_test_split["train"]
temp_ds = train_test_split["test"]

# Second split: 20% test, 30% val (out of total)
val_test_split = temp_ds.train_test_split(test_size=2/3, seed=42)  # 1/3 val, 2/3 test → overall 30/20
val_ds = val_test_split["train"]
test_ds = val_test_split["test"]

# Combine
ds_split = {
    "train": train_ds,
    "validation": val_ds,
    "test": test_ds
}

print(f"Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}")

Train: 26001 | Val: 8667 | Test: 17334


In [27]:
print(val_ds[0])

{'instruction': 'Give a summary of the recent US presidential election.', 'input': '', 'output': "The 2020 US Presidential Election was the 59th quadrennial presidential election. Incumbent President Donald Trump of the Republican Party ran against the Democratic Party's nominee, former Vice President Joe Biden. Biden won the November 3rd election, having received 306 electoral votes to Trump's 232. He was certified as the winner on December 14th, 20", 'text': "Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nGive a summary of the recent US presidential election.\n\n### Response:\nThe 2020 US Presidential Election was the 59th quadrennial presidential election. Incumbent President Donald Trump of the Republican Party ran against the Democratic Party's nominee, former Vice President Joe Biden. Biden won the November 3rd election, having received 306 electoral votes to Trump's 232. He was certified as the winne

In [28]:
def format_input(entry):
    instruction_text = (
    f"Below is an instruction that describes a task. "
    f"Write a response that appropriately completes the request."
    f"\n\n### Instruction:\n{entry['instruction']}"
    )
    input_text = (
    f"\n\n### Input:\n{entry['input']}" if entry["input"] else ""
    )
    output_text = f"\n\n### Response:\n{entry['output']}" if entry['output'] else ""
    return instruction_text + input_text + output_text

In [29]:
train_ds[0]

{'instruction': 'Besides the text inputed before, provide an additional input to this instruction.',
 'input': 'Obama has announced his plan to end the Iraq war.',
 'output': '<noinput>',
 'text': 'Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.\n\n### Instruction:\nBesides the text inputed before, provide an additional input to this instruction.\n\n### Input:\nObama has announced his plan to end the Iraq war.\n\n### Response:\n<noinput>'}

In [30]:
model_input = format_input(train_ds[0])
model_input

'Below is an instruction that describes a task. Write a response that appropriately completes the request.\n\n### Instruction:\nBesides the text inputed before, provide an additional input to this instruction.\n\n### Input:\nObama has announced his plan to end the Iraq war.\n\n### Response:\n<noinput>'

In [31]:
class InstructionDataset(Dataset):
    def __init__(self, data, tokenizer):
        self.data = data
        self.encoded_texts = []
        for entry in data:
            full_text = format_input(entry)
            # response_text = f"\n\n### Response:\n{entry['output']}"
            # full_text = instruction_plus_input + response_text
            self.encoded_texts.append(
            tokenizer.encode(full_text)
            )
    def __getitem__(self, index):
        return self.encoded_texts[index]
    def __len__(self):
        return len(self.data)

In [32]:
import tiktoken
tokenizer = tiktoken.get_encoding("gpt2")
print(tokenizer.encode("<|endoftext|>", allowed_special={"<|endoftext|>"}))

[50256]


In [ ]:
from typing import List
import torch

def custom_collate_fn(
    batch: List[List[int]],
    pad_token_id: int = 50256,
    ignore_index: int = -100,
    allowed_max_length: int | None = None
):
    """
    batch: list of token id sequences (python lists) for causal LM.
    Appends an EOS/pad, does next-token prediction shift, and pads to max length in the batch.
    Returns a dict suitable for GPT-2: input_ids, attention_mask, labels (pads -> -100).
    """
    # +1 for the extra EOS/pad we append
    batch_max_len = max(len(item) + 1 for item in batch)
    if allowed_max_length is not None:
        batch_max_len = min(batch_max_len, allowed_max_length)

    input_rows, label_rows = [], []

    for item in batch:
        # copy -> append one EOS/pad at the end
        seq = list(item)
        seq.append(pad_token_id)

        # optionally truncate BEFORE building shifted pairs
        if len(seq) > batch_max_len:
            seq = seq[:batch_max_len]

        # right-pad with pad_token_id up to batch_max_len
        if len(seq) < batch_max_len:
            seq = seq + [pad_token_id] * (batch_max_len - len(seq))

        # causal LM: inputs are tokens[:-1], labels are tokens[1:]
        inp = torch.tensor(seq[:-1], dtype=torch.long)
        tgt = torch.tensor(seq[1:], dtype=torch.long)

        input_rows.append(inp)
        label_rows.append(tgt)

    input_ids = torch.stack(input_rows, dim=0)          # (B, T)
    labels    = torch.stack(label_rows, dim=0)          # (B, T)

    # attention_mask: 1 where not pad
    attention_mask = (input_ids != pad_token_id).long()

    # ignore pads in labels
    labels[attention_mask == 0] = ignore_index

    return {
        "input_ids": input_ids,                  # Long (B, T)
        "attention_mask": attention_mask,        # Long (B, T)
        "labels": labels,                        # Long (B, T) with pads=-100
    }

In [34]:
inputs_1 = [0, 1, 2, 3, 4]
inputs_2 = [5, 6]
inputs_3 = [7, 8, 9]
batch = [
inputs_1,
inputs_2,
inputs_3]
out = custom_collate_fn(batch)

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [39]:
device

device(type='cuda')

In [46]:
from functools import partial
customized_collate_fn = partial(
    custom_collate_fn,
    allowed_max_length=1024
)

In [57]:
num_workers = 4
batch_size = 32
torch.manual_seed(123)
train_dataset = InstructionDataset(train_ds, tokenizer)
train_loader = DataLoader(
train_dataset,
batch_size=batch_size,
collate_fn=customized_collate_fn,
shuffle=True,
drop_last=True,
num_workers=num_workers
)
val_dataset = InstructionDataset(val_ds, tokenizer)
val_loader = DataLoader(
val_dataset,
batch_size=batch_size,
collate_fn=customized_collate_fn,
shuffle=False,
drop_last=False,
num_workers=num_workers
)
test_dataset = InstructionDataset(test_ds, tokenizer)
test_loader = DataLoader(
test_dataset,
batch_size=batch_size,
collate_fn=customized_collate_fn,
shuffle=False,
drop_last=False,
num_workers=num_workers
)

In [58]:
for batch in train_loader:
    print(batch['input_ids'].shape,batch['labels'].shape)

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

torch.Size([32, 306]) torch.Size([32, 306])
torch.Size([32, 294]) torch.Size([32, 294])
torch.Size([32, 318]) torch.Size([32, 318])
torch.Size([32, 266]) torch.Size([32, 266])
torch.Size([32, 336]) torch.Size([32, 336])
torch.Size([32, 426]) torch.Size([32, 426])
torch.Size([32, 342]) torch.Size([32, 342])
torch.Size([32, 210]) torch.Size([32, 210])
torch.Size([32, 245]) torch.Size([32, 245])
torch.Size([32, 196]) torch.Size([32, 196])
torch.Size([32, 363]) torch.Size([32, 363])
torch.Size([32, 299]) torch.Size([32, 299])
torch.Size([32, 236]) torch.Size([32, 236])
torch.Size([32, 391]) torch.Size([32, 391])
torch.Size([32, 203]) torch.Size([32, 203])
torch.Size([32, 183]) torch.Size([32, 183])
torch.Size([32, 221]) torch.Size([32, 221])
torch.Size([32, 178]) torch.Size([32, 178])
torch.Size([32, 200]) torch.Size([32, 200])
torch.Size([32, 239]) torch.Size([32, 239])
torch.Size([32, 283]) torch.Size([32, 283])
torch.Size([32, 400]) torch.Size([32, 400])
torch.Size([32, 328]) torch.Size

In [ ]:
# import math, os, torch
# from torch.optim import AdamW
# from transformers import (
#     GPT2LMHeadModel,
#     GPT2TokenizerFast,
#     get_linear_schedule_with_warmup,
# )

# # -------------------
# # device & reproducibility
# # -------------------
# torch.manual_seed(123)
# device = (
#     "cuda" if torch.cuda.is_available()
#     else "mps" if torch.backends.mps.is_available()
#     else "cpu"
# )

# # -------------------
# # tokenizer & model
# # (ensure PAD token exists for batching)
# # -------------------
# model_name = "gpt2"  # or "gpt2-medium"/custom
# tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
# if tokenizer.pad_token is None:
#     tokenizer.pad_token = tokenizer.eos_token

# model = GPT2LMHeadModel.from_pretrained(model_name)
# model.resize_token_embeddings(len(tokenizer))
# model.config.pad_token_id = tokenizer.pad_token_id
# model.to(device)

# # -------------------
# # training knobs
# # -------------------
# epochs = 2
# lr = 5e-5
# weight_decay = 0.01
# max_grad_norm = 1.0
# grad_accum_steps = 1  # keep 1 unless you want bigger effective batch
# warmup_ratio = 0.03   # small warmup helps stability

# # your existing loaders (already built)
# # train_loader, val_loader, test_loader
# # each batch should be a dict with at least: input_ids, attention_mask, labels
# # labels should use -100 for pads (ignore_index)

# # -------------------
# # optimizer & scheduler
# # -------------------
# no_decay = ["bias", "LayerNorm.weight"]
# param_groups = [
#     {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
#      "weight_decay": weight_decay},
#     {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
#      "weight_decay": 0.0},
# ]
# optimizer = AdamW(param_groups, lr=lr)

# num_update_steps_per_epoch = math.ceil(len(train_loader) / grad_accum_steps)
# num_training_steps = epochs * num_update_steps_per_epoch
# num_warmup_steps = int(warmup_ratio * num_training_steps)

# scheduler = get_linear_schedule_with_warmup(
#     optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps
# )

# # -------------------
# # helpers
# # -------------------
# def run_epoch(dataloader, train: bool):
#     model.train(train)
#     total_loss, steps = 0.0, 0
#     if train:
#         optimizer.zero_grad(set_to_none=True)

#     for step, batch in enumerate(dataloader):
#         batch = {k: v.to(device) for k, v in batch.items()}
#         # Passing labels lets the model compute CE with ignore_index=-100 internally
#         outputs = model(**batch)
#         loss = outputs.loss / grad_accum_steps

#         if train:
#             loss.backward()
#             if (step + 1) % grad_accum_steps == 0:
#                 torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
#                 optimizer.step()
#                 scheduler.step()
#                 optimizer.zero_grad(set_to_none=True)

#         total_loss += loss.item()
#         steps += 1

#     avg_loss = total_loss / max(steps, 1)
#     ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")  # guard overflow
#     return avg_loss, ppl

# # -------------------
# # train / val loop + checkpoint best
# # -------------------
# best_val_loss = float("inf")
# save_dir = "ckpt_gpt2_best"
# os.makedirs(save_dir, exist_ok=True)

# for epoch in range(1, epochs + 1):
#     train_loss, train_ppl = run_epoch(train_loader, train=True)
#     val_loss, val_ppl = run_epoch(val_loader, train=False)

#     print(f"[Epoch {epoch}] "
#           f"train_loss={train_loss:.4f} train_ppl={train_ppl:.2f} | "
#           f"val_loss={val_loss:.4f} val_ppl={val_ppl:.2f}")

#     if val_loss < best_val_loss:
#         best_val_loss = val_loss
#         model.save_pretrained(save_dir)
#         tokenizer.save_pretrained(save_dir)
#         print(f"  -> saved best checkpoint to {save_dir}")



/home/ramsred/mmml/mmml-venv/lib/python3.12/site-packages/torch/cuda/__init__.py:283: UserWarning: 
    Found GPU0 NVIDIA GB10 which is of cuda capability 12.1.
    Minimum and Maximum cuda capability supported by this version of PyTorch is
    (8.0) - (12.0)
    
  warnings.warn(
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


KeyboardInterrupt: 

In [59]:
import math, os, torch
from torch.optim import AdamW
from transformers import (
    GPT2LMHeadModel,
    GPT2TokenizerFast,
    get_linear_schedule_with_warmup,
)
from tqdm import tqdm  # NEW

# -------------------
# device & reproducibility
# -------------------
torch.manual_seed(123)
device = (
    "cuda" if torch.cuda.is_available()
    else "mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available()
    else "cpu"
)
print(f"Using device: {device}")

# -------------------
# tokenizer & model
# (ensure PAD token exists for batching)
# -------------------
model_name = "gpt2"  # or "gpt2-medium"/custom
tokenizer = GPT2TokenizerFast.from_pretrained(model_name)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained(model_name)
model.resize_token_embeddings(len(tokenizer))
model.config.pad_token_id = tokenizer.pad_token_id
model.to(device)

# -------------------
# training knobs
# -------------------
epochs = 2
lr = 5e-5
weight_decay = 0.01
max_grad_norm = 1.0
grad_accum_steps = 1  # keep 1 unless you want bigger effective batch
warmup_ratio = 0.03   # small warmup helps stability

# your existing loaders (already built)
# train_loader, val_loader, test_loader
# each batch should be a dict with at least: input_ids, attention_mask, labels
# labels should use -100 for pads (ignore_index)

# -------------------
# optimizer & scheduler
# -------------------
no_decay = ["bias", "LayerNorm.weight"]
param_groups = [
    {"params": [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     "weight_decay": weight_decay},
    {"params": [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
     "weight_decay": 0.0},
]
optimizer = AdamW(param_groups, lr=lr)

num_update_steps_per_epoch = math.ceil(len(train_loader) / grad_accum_steps)
num_training_steps = epochs * num_update_steps_per_epoch
num_warmup_steps = int(warmup_ratio * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=num_warmup_steps, num_training_steps=num_training_steps
)

# -------------------
# helpers
# -------------------
def run_epoch(dataloader, train: bool, desc: str = None):
    model.train(train)
    total_loss, steps = 0.0, 0
    if train:
        optimizer.zero_grad(set_to_none=True)

    # tqdm progress bar
    loop = tqdm(dataloader, desc=desc or ("Training" if train else "Evaluating"), leave=False)
    for step, batch in enumerate(loop):
        batch = {k: v.to(device) for k, v in batch.items()}
        # Passing labels lets the model compute CE with ignore_index=-100 internally
        outputs = model(**batch)
        loss = outputs.loss / grad_accum_steps

        if train:
            loss.backward()
            if (step + 1) % grad_accum_steps == 0:
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_grad_norm)
                optimizer.step()
                scheduler.step()
                optimizer.zero_grad(set_to_none=True)

        total_loss += loss.item()
        steps += 1

        # Update progress bar with the (per-step) loss
        loop.set_postfix(loss=float(loss.item()))

    avg_loss = total_loss / max(steps, 1)
    ppl = math.exp(avg_loss) if avg_loss < 20 else float("inf")  # guard overflow
    return avg_loss, ppl

# -------------------
# train / val loop + checkpoint best
# -------------------
best_val_loss = float("inf")
save_dir = "ckpt_gpt2_best"
os.makedirs(save_dir, exist_ok=True)

for epoch in range(1, epochs + 1):
    train_loss, train_ppl = run_epoch(train_loader, train=True,  desc=f"Train Epoch {epoch}")
    val_loss,   val_ppl   = run_epoch(val_loader,   train=False, desc=f"Eval  Epoch {epoch}")

    print(f"[Epoch {epoch}] "
          f"train_loss={train_loss:.4f} train_ppl={train_ppl:.2f} | "
          f"val_loss={val_loss:.4f} val_ppl={val_ppl:.2f}")

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        model.save_pretrained(save_dir)
        tokenizer.save_pretrained(save_dir)
        print(f"  -> saved best checkpoint to {save_dir}")



Using device: cuda


Train Epoch 1:   0%|          | 0/812 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_P

[Epoch 1] train_loss=3.3779 train_ppl=29.31 | val_loss=2.8659 val_ppl=17.56
  -> saved best checkpoint to ckpt_gpt2_best


Train Epoch 2:   0%|          | 0/812 [00:00<?, ?it/s]huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_P

[Epoch 2] train_loss=2.8649 train_ppl=17.55 | val_loss=2.7945 val_ppl=16.36
  -> saved best checkpoint to ckpt_gpt2_best


In [ ]:
# (Optional) test at the end
test_loss, test_ppl = run_epoch(test_loader, train=False, desc="Testing")
print(f"[Test] loss={test_loss:.4f} ppl={test_ppl:.2f}")

In [60]:
# -------------------
# load best & test
# -------------------
best_model = GPT2LMHeadModel.from_pretrained(save_dir).to(device)
best_model.eval()
with torch.no_grad():
    total, steps = 0.0, 0
    for batch in test_loader:
        batch = {k: v.to(device) for k, v in batch.items()}
        loss = best_model(**batch).loss
        total += loss.item(); steps += 1
    test_loss = total / max(steps, 1)
    test_ppl = math.exp(test_loss) if test_loss < 20 else float("inf")
print(f"[TEST] loss={test_loss:.4f} ppl={test_ppl:.2f}")

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

[TEST] loss=2.7831 ppl=16.17


In [61]:
def quick_generate(model, tok, prompt="In a small research lab, a curious student discovered", max_new_tokens=60):
    model.eval()
    device = next(model.parameters()).device
    inputs = tok(prompt, return_tensors="pt").to(device)
    pad_id = tok.pad_token_id if tok.pad_token_id is not None else tok.eos_token_id
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            do_sample=True, top_p=0.95, temperature=0.9,
            pad_token_id=pad_id,
        )
    print("\n=== SAMPLE GENERATION ===")
    print(tok.decode(out[0], skip_special_tokens=True))
    print("=========================\n")

In [ ]:
query = "tell me something about penguins?"
quick_generate(best_model, tokenizer,prompt=query)


=== SAMPLE GENERATION ===
How are you doing today?

#���������������������������������������������������������

